<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 4: Case D and Multiple Objectives

**A study plan can reduce the learning gap, but doing so usually uses more time.**

Part 3 classified decisions by their allowed values. This part formulates the study-time case from 01-2 and distinguishes raw performance, a scalar objective, and a vector objective.

### 1 · Carry forward the study decision

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_d_study.png" alt="Math and writing notebooks share a desk with a clock and a planner marked Up to 4 hours." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case D from 01-2: math hours \(m\) and writing hours \(w\) form \(x=[m,w]^{\mathsf T}\). The formulation compares learning gap and total study time.

Let \(m\) and \(w\) be mathematics and writing study hours. The student has at most 4 hours. Two performance outputs are

> $\displaystyle Q(m,w)=\frac{20}{1+m}+\frac{15}{1+w},\qquad S(m,w)=m+w.$

The learning gap \(Q\) becomes smaller with study. The time use \(S\) becomes larger. The same decision can therefore improve one output and worsen the other.

### 2 · Write the multi-objective formulation

Let \(x=[m,w]^{\mathsf T}\) and \(y=\operatorname{Sim}(x)=[Q(x),S(x)]^{\mathsf T}\). Here \(\operatorname{Sim}\) is a direct algebraic response calculation.

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad
\boldsymbol f(y)=\begin{bmatrix}Q(x)\\S(x)\end{bmatrix}$
>
> $\displaystyle \text{subject to}\quad -m\le0,\quad -w\le0,\quad m+w-4\le0.$

Uppercase \(F\) is not used for the vector objective. The notation \(\boldsymbol f\) distinguishes several objective values from the scalar \(f\).

A feasible plan \(x^A\) **dominates** another feasible plan \(x^B\) when it is no worse in both objectives and strictly better in at least one. A Pareto candidate is not dominated by another feasible candidate.

### 3 · Connect the study decision to its two objectives

The left panel locates the hours \((m,w)\). The teal triangle satisfies the four-hour limit, and blue curves connect plans with equal learning gap \(Q\). The right panel locates the resulting pair \((Q,S)\).

The two orange markers refer to the same plan. In the performance plot, dark teal points are nondominated grid candidates, while light teal points are other feasible grid candidates. Both objectives prefer the lower-left direction, so a vector objective alone does not identify one preferred plan.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_d_formulation_graph.png" alt="A study-hour decision plane and a learning-gap versus time plot show how one study plan produces two competing objective values." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
import numpy as np

MAX_STUDY_TIME = 4.0


def evaluate_study_plan(x):
    math_hours, writing_hours = np.asarray(x, dtype=float)
    learning_gap = 20.0 / (1.0 + math_hours) + 15.0 / (1.0 + writing_hours)
    study_time = math_hours + writing_hours
    residuals = np.array([-math_hours, -writing_hours, study_time - MAX_STUDY_TIME])
    return {
        "x": (math_hours, writing_hours),
        "y": (learning_gap, study_time),
        "g": residuals,
        "feasible": bool(np.all(residuals <= 1e-10)),
    }


def pareto_front(records):
    feasible = [record for record in records if record["feasible"]]
    nondominated = []
    for candidate in feasible:
        q_value, s_value = candidate["y"]
        dominated = any(
            other["y"][0] <= q_value
            and other["y"][1] <= s_value
            and (other["y"][0] < q_value or other["y"][1] < s_value)
            for other in feasible
        )
        if not dominated:
            nondominated.append(candidate)
    return sorted(nondominated, key=lambda record: record["y"][1])

In [ ]:
import sys
import matplotlib


def _pyplot(*, interactive=False):
    """Use the course's widget-backend fallback outside the browser runtime."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
PURPLE = "#8856a7"
GRAY = "#a6a6a6"
GOLD = "#f6c945"


def case_canvas(title, controls, *, panels=2, interactive=False):
    """Create a shared layout; controls are (name, label, low, high, value, step, color)."""
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(
        1, panels, figsize=(12.4 if panels == 2 else 14.0, 7.4 if interactive else 5.3)
    )
    figure.suptitle(title, y=0.985, fontsize=14, fontweight="bold")
    status = figure.text(0.5, 0.918, "", ha="center", va="center", fontsize=11)
    footer = figure.text(0.5, 0.045 if not interactive else 0.27, "",
                         ha="center", va="center", fontsize=9)
    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider

        figure.subplots_adjust(left=0.075, right=0.97,
                               bottom=0.41 if panels == 3 else 0.37, top=0.83, wspace=0.38)
        positions = np.linspace(0.19, 0.065, max(len(controls), 2))
        for position, (name, label, low, high, value, step, color) in zip(positions, controls):
            slider_axis = figure.add_axes([0.28, position, 0.61, 0.026])
            sliders[name] = Slider(slider_axis, label, low, high, valinit=value,
                                   valstep=step, valfmt="%1.0f" if step >= 1 else "%1.2f",
                                   color=color, initcolor=color)
    figure._case_sliders = sliders
    figure._case_state = {}
    return plt, figure, axes, sliders, status, footer


def finish_case(plt, figure, axes, sliders, refresh, *, interactive=False):
    """Connect controls and keep widgets and evaluated records alive on the figure."""
    for axis in axes:
        axis.grid(alpha=0.25)
    for slider in sliders.values():
        slider.on_changed(refresh)
    figure._case_refresh = refresh
    refresh()
    if not interactive:
        figure.tight_layout(rect=(0.015, 0.09, 0.985, 0.96))
    plt.show()
    if not interactive:
        plt.close(figure)
    return figure


def show_study_formulation(decision=(1.0, 0.5), weight=None, *, interactive=False):
    controls = [
        ("math", "Decision: math m (h)", 0.0, MAX_STUDY_TIME, decision[0], 0.25, BLUE),
        ("writing", "Decision: writing w (h)", 0.0, MAX_STUDY_TIME, decision[1], 0.25, BLUE),
    ]
    if weight is not None:
        controls.append(("weight", "Preference: learning-gap weight", 0.0, 1.0, weight, 0.05, PURPLE))
    plt, figure, axes, sliders, status, footer = case_canvas(
        "Case D · A study decision becomes a learning-gap / time trade-off",
        controls, interactive=interactive,
    )
    from matplotlib.patches import Polygon

    levels = np.arange(0.0, MAX_STUDY_TIME + 0.125, 0.25)
    records = [evaluate_study_plan([m, w]) for m in levels for w in levels]
    feasible = [record for record in records if record["feasible"]]
    front = pareto_front(records)
    axes[0].set_facecolor("#f0f0f0")
    axes[0].add_patch(Polygon([(0, 0), (4, 0), (0, 4)], color=TEAL, alpha=0.18,
                              label="Feasible study plans"))
    mesh = np.linspace(0, 4, 81)
    math_hours, writing_hours = np.meshgrid(mesh, mesh)
    gap = 20.0 / (1 + math_hours) + 15.0 / (1 + writing_hours)
    contours = axes[0].contour(math_hours, writing_hours, gap,
                               levels=[10, 12, 15, 20, 25, 30], colors=BLUE, linewidths=0.8)
    axes[0].clabel(contours, fontsize=8, fmt="%d")
    axes[0].plot([0, 4], [4, 0], "--", color="#555555", label="Total study time = 4 h")
    decision_marker = axes[0].scatter([], [], color=ORANGE, edgecolor="black", s=100,
                                      label="Current plan", zorder=5)
    axes[0].set(xlabel="Math study m (h)", ylabel="Writing study w (h)",
                xlim=(-0.15, 4.15), ylim=(-0.15, 5.3), title="Blue lines: equal learning gap Q")
    axes[0].legend(fontsize=8, loc="upper right")
    axes[1].scatter([r["y"][0] for r in feasible], [r["y"][1] for r in feasible],
                    color="#80cbc4", alpha=0.6, s=25, label="Feasible grid plans")
    axes[1].plot([r["y"][0] for r in front], [r["y"][1] for r in front],
                 color=TEAL, marker=".", linewidth=2, label="Nondominated grid plans")
    performance_marker = axes[1].scatter([], [], color=ORANGE, edgecolor="black", s=100,
                                         zorder=5, label="Current plan")
    axes[1].set(xlabel="Learning gap Q (gap units)", ylabel="Study time S (h)",
                xlim=(6, 37), ylim=(-0.2, 4.7), title="Moving the decision moves both outputs")
    if weight is not None:
        selected_decision = axes[0].scatter([], [], marker="*", s=230, color=GOLD,
                                             edgecolor="black", zorder=6)
        selected_performance = axes[1].scatter([], [], marker="*", s=230, color=GOLD,
                                                edgecolor="black", zorder=6,
                                                label="Best weighted grid plan")
    axes[1].legend(fontsize=8, loc="upper right")

    def refresh(_=None):
        values = (sliders["math"].val, sliders["writing"].val) if sliders else decision
        alpha = sliders["weight"].val if "weight" in sliders else weight
        current = evaluate_study_plan(values)
        learning_gap, study_time = current["y"]
        decision_marker.set_offsets([current["x"]])
        performance_marker.set_offsets([current["y"]])
        axes[1].set_ylim(-0.2, max(4.7, study_time + 0.5))
        eligibility = "FEASIBLE" if current["feasible"] else "REJECTED: more than 4 h"
        status.set_text(f"Current (m, w) = ({values[0]:.2f}, {values[1]:.2f}) h"
                        f"   |   Q = {learning_gap:.2f}, S = {study_time:.2f} h   |   {eligibility}")
        status.set_color(TEAL if current["feasible"] else "#555555")
        selected = None
        current_score = None
        if alpha is not None:
            def score(record):
                gap, hours = record["y"]
                return alpha * gap / 35.0 + (1.0 - alpha) * hours / 4.0
            selected = min(feasible, key=score)
            current_score = score(current)
            selected_decision.set_offsets([selected["x"]])
            selected_performance.set_offsets([selected["y"]])
            footer.set_text(f"Weight = {alpha:.2f}: f = weight × Q/35 + (1 - weight) × S/4."
                            f"   Current f = {current_score:.3f}; best feasible grid f = {score(selected):.3f}.\n"
                            "Changing the purple weight moves the gold selection; the orange plan and its two outputs stay fixed.")
        else:
            footer.set_text("The two objectives prefer the lower-left corner. The teal curve shows trade-offs on the 0.25-h grid.")
        figure._case_state.update(current=current, selected=selected, weight=alpha,
                                   current_score=current_score, pareto=front)
        figure.canvas.draw_idle()

    return finish_case(plt, figure, axes, sliders, refresh, interactive=interactive)

In [ ]:
static_figure = show_study_formulation(decision=(1.0, 0.5))

The teal curve is the Pareto front of the stated finite grid, not a proof of the continuous Pareto front. Moving along it exchanges learning gap for study time. The vector objective identifies trade-offs but does not select one plan by itself.

### 4 · Add a stated selection rule when one plan is needed

A weighted sum creates a scalar objective:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad
f(y;\alpha)=\alpha\,\widetilde Q(x)+(1-\alpha)\,\widetilde S(x),
\qquad 0\le\alpha\le1.$

Here, \(\widetilde Q=Q/35\) uses the no-study learning gap as a reference, and \(\widetilde S=S/4\) uses the four-hour allowance. Both ratios are dimensionless. The following trackbars use these fixed scales. The weight \(\alpha\) is a hyperparameter; it changes the comparison rule, not the outcomes of a fixed study plan.

The blue trackbars change \(m\) and \(w\). Their orange markers move together in decision space and performance space. Plans above the four-hour limit are rejected.

The purple trackbar changes \(\alpha\) in the scalar score defined above. Its gold stars mark the best feasible 0.25-hour grid plan in both panels. Changing only \(\alpha\) can move the gold stars while the current orange plan and its raw outputs remain fixed.

In [ ]:
formulation_explorer = show_study_formulation(
    decision=(1.0, 0.5), weight=0.5, interactive=True
)

An alternative keeps one objective and turns the other into a requirement:

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad Q(x)
\qquad\text{subject to}\qquad S(x)\le\varepsilon.$

Changing \(\alpha\) changes the scalar score. Changing \(\varepsilon\) changes the feasible set. These are different formulations.

### 5 · Classify Case D

Case D is a **generally constrained, continuous, multi-objective, nonlinear, direct algebraic, deterministic optimization problem**. Choosing a weighted sum would change only its objective classification to single-objective.

### Takeaway

Count the values that the formulation asks us to optimize:

> **one scalar \(f\) → single-objective · vector \(\boldsymbol f\) → multi-objective · dominance removes inferior choices · a preference or requirement selects one trade-off**

Performance outputs describe a plan. The objective supplies the comparison rule. Part 5 keeps these roles visible while changing function structure, response evaluation, and treatment of uncertain weather.